In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_info_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateInfoId String, SourceSystem String,"
        #     "IngestionTimestamp timestamp, BatchId String, RealEstateInfoKey string, RealEstateKey String, OwnerPartyKey String, OwnershipKey string, OwnershipReferenceKey String,"
        #     "OwnershipCode String, OwnershipTypeCode String, OwnershipType String, OwnershipPercentage Double,"
        #     "OwnershipStatusCode String, OwnershipStatus String, OwnershipStatusDate Date, NotaryDate Date, NotaryNumber String,"
        #     "RELienKey String, RELienReferenceKey String, RELienIsActiveFlag Boolean, RELienStatusCode String, RELienStatus String,"
        #     "RELienActive Boolean, RELienTypeCode String, RELienType String, RELienReasonCode String,"
        #     "RELienReason String, RELienInitialOrder Int, RELienCurrentOrder Int, RELienCreditorKey String, RELienCreditorName String,"
        #     "RELienCreditorType string, RELienMortgageTypeCode String, RELienMortgageType String, RELienMortgageAmount Decimal(19,4), "
        #     "RELienCurrency String, RELienComments String, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateInfo"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== eaiSIG_REPW ==================

        eaiSIG_REPW_extra_filters = None

        if eaiSIG_REPW_extra_filters is not None:
            eaiSIG_REPW_extra_filters.append(date_filter)
            eaiSIG_REPW_final_filters = " AND ".join(eaiSIG_REPW_extra_filters)
        else:
            eaiSIG_REPW_final_filters = date_filter

        eaiSIG_REPW_selected_fields = [
            "eaiSIG_REPW_CODE",
            "eaiSIG_REPW_REAL_ESTATE_CODE",
            "eaiSIG_REPW_DATE",
            "eaiSIG_REPW_KEY",
            "eaiSIG_REPW_TYPE",
            "eaiSIG_REPW_UNDERWRITING_ORDER",
            "eaiSIG_REPW_BANK_NAME",
            "eaiSIG_REPW_AMOUNT",
            "eaiSIG_REPW_CURRENCY",
            "businessdate",
        ]

        eaiSIG_REPW = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_REPW",
            eaiSIG_REPW_final_filters,
            eaiSIG_REPW_selected_fields,
        )

        # ================== eaiSIG_REP ==================

        eaiSIG_REP_extra_filters = None

        if eaiSIG_REP_extra_filters is not None:
            eaiSIG_REP_extra_filters.append(date_filter)
            eaiSIG_REP_final_filters = " AND ".join(eaiSIG_REP_extra_filters)
        else:
            eaiSIG_REP_final_filters = date_filter

        eaiSIG_REP_selected_fields = [
            "eaiSIG_REP_CUSTOMER_ID",
            "eaiSIG_REP_REAL_ESTATE_CODE",
            "eaiSIG_REP_PROPERTY_CODE",
            "businessdate",
        ]

        eaiSIG_REP = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_REP",
            eaiSIG_REP_final_filters,
            eaiSIG_REP_selected_fields,
        )

        # ================== eaiSIG_RE ==================

        eaiSIG_RE_extra_filters = None

        if eaiSIG_RE_extra_filters is not None:
            eaiSIG_RE_extra_filters.append(date_filter)
            eaiSIG_RE_final_filters = " AND ".join(eaiSIG_RE_extra_filters)
        else:
            eaiSIG_RE_final_filters = date_filter

        eaiSIG_RE_selected_fields = [
            "eaiSIG_RE_OWNERSHIP_TYPE",
            "eaiSIG_RE_REAL_ESTATE_CODE",
            "businessdate",
        ]

        eaiSIG_RE = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_RE",
            eaiSIG_RE_final_filters,
            eaiSIG_RE_selected_fields,
        )

        # ================== Joins ==================

        eaiSIG_REPW_eaiSIG_REP_join_condition = (
            eaiSIG_REPW["eaiSIG_REPW_REAL_ESTATE_CODE"]
            == eaiSIG_REP["eaiSIG_REP_REAL_ESTATE_CODE"]
        )

        eaiSIG_RE_eaiSIG_REP_join_condition = (
            eaiSIG_RE["eaiSIG_RE_REAL_ESTATE_CODE"]
            == eaiSIG_REP["eaiSIG_REP_REAL_ESTATE_CODE"]
        )

        target_table_arotron = eaiSIG_REPW.join(
            eaiSIG_REP, eaiSIG_REPW_eaiSIG_REP_join_condition, "inner"
        ).join(eaiSIG_RE, eaiSIG_RE_eaiSIG_REP_join_condition, "inner")

        target_table_arotron_RealEstateInfoId_primary_key_fields = [
            eaiSIG_REPW.eaiSIG_REPW_CODE,
            eaiSIG_REP.eaiSIG_REP_CUSTOMER_ID,
            eaiSIG_REP.eaiSIG_REP_PROPERTY_CODE,
            lit("arotron"),
        ]
        RealEstateInfoKey_primary_key_fields = [
            eaiSIG_REPW.eaiSIG_REPW_CODE,
            eaiSIG_REP.eaiSIG_REP_CUSTOMER_ID,
            eaiSIG_REP.eaiSIG_REP_PROPERTY_CODE,
        ]

        target_table_arotron = (
            target_table_arotron.withColumn(
                "RealEstateInfoId",
                sha2(
                    concat_ws(
                        ";", *target_table_arotron_RealEstateInfoId_primary_key_fields
                    ),
                    256,
                ),
            )
            .withColumn(
                "RealEstateInfoKey",
                concat_ws(";", *RealEstateInfoKey_primary_key_fields),
            )
            .withColumn(
                "RELienCurrentOrder",
                when(eaiSIG_REPW["eaiSIG_REPW_UNDERWRITING_ORDER"] == "", lit(None))
                .otherwise(
                    substring(eaiSIG_REPW["eaiSIG_REPW_UNDERWRITING_ORDER"], 1, 2)
                )
                .cast("int"),
            )
            .withColumn(
                "RELienMortgageAmount",
                when(eaiSIG_REPW["eaiSIG_REPW_AMOUNT"] == "", lit(None))
                .otherwise(substring(eaiSIG_REPW["eaiSIG_REPW_AMOUNT"], 2, 100))
                .cast("decimal(19,4)"),
            )
            .withColumn(
                "NotaryDate",
                when(eaiSIG_REPW["eaiSIG_REPW_DATE"] == "", lit(None)).otherwise(
                    eaiSIG_REPW["eaiSIG_REPW_DATE"].cast("date")
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.RealEstateInfoId.alias(
                "RealEstateInfoId"
            ),  # problem, the field is actually of the "String" data type for the arotron source system
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            target_table_arotron.RealEstateInfoKey.cast("String").alias(
                "RealEstateInfoKey"
            ),
            eaiSIG_REPW.eaiSIG_REPW_REAL_ESTATE_CODE.cast("string").alias(
                "RealEstateKey"
            ),  # problem, the field is actually of the "String" data type for the arotron source system
            eaiSIG_REP.eaiSIG_REP_CUSTOMER_ID.cast("string").alias(
                "OwnerPartyKey"
            ),  # problem, the field is actually of the "String" data type for the arotron source system
            lit(None).cast("string").alias("OwnershipKey"),
            lit(None).cast("String").alias("OwnershipReferenceKey"),
            lit(None).cast("String").alias("OwnershipCode"),
            eaiSIG_RE.eaiSIG_RE_OWNERSHIP_TYPE.cast("String").alias(
                "OwnershipTypeCode"
            ),  # Does not have description
            eaiSIG_RE.eaiSIG_RE_OWNERSHIP_TYPE.cast("String").alias(
                "OwnershipType"
            ),  # Same as above as there is no description
            lit(None).cast("Double").alias("OwnershipPercentage"),
            lit(None).cast("String").alias("OwnershipStatusCode"),
            lit(None).cast("String").alias("OwnershipStatus"),
            lit(None).cast("Date").alias("OwnershipStatusDate"),
            target_table_arotron.NotaryDate.alias(
                "NotaryDate"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but we convert it above to the date format.
            lit(None).cast("String").alias("NotaryNumber"),
            eaiSIG_REPW.eaiSIG_REPW_CODE.cast("string").alias(
                "RELienKey"
            ),  # problem, the field is actually of the "String" data type for the arotron source system. The field contains only null values so this would be a problem.
            lit(None).cast("String").alias("RELienReferenceKey"),
            lit(None).cast("Boolean").alias("RELienIsActiveFlag"),
            lit(None).cast("String").alias("RELienStatusCode"),
            lit(None).cast("String").alias("RELienStatus"),
            lit(None).cast("Boolean").alias("RELienActive"),
            eaiSIG_REPW.eaiSIG_REPW_TYPE.cast("String").alias(
                "RELienTypeCode"
            ),  # It is not used in any procedure
            lit(None).cast("String").alias("RELienType"),
            lit(None).cast("String").alias("RELienReasonCode"),
            lit(None).cast("String").alias("RELienReason"),
            lit(None).cast("Int").alias("RELienInitialOrder"),
            target_table_arotron.RELienCurrentOrder.alias(
                "RELienCurrentOrder"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to integer works. The blank values however get converted to 0.
            lit(None).cast("string").alias("RELienCreditorKey"),
            lit(None).cast("String").alias("RELienCreditorName"),
            lit(None).cast("String").alias("RELienCreditorType"),
            eaiSIG_REPW.eaiSIG_REPW_TYPE.cast("String").alias(
                "RELienMortgageTypeCode"
            ),  # It is not used in any procedure
            lit(None).cast("String").alias("RELienMortgageType"),
            target_table_arotron.RELienMortgageAmount.alias(
                "RELienMortgageAmount"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            eaiSIG_REPW.eaiSIG_REPW_CURRENCY.cast("String").alias("RELienCurrency"),
            lit(None).cast("String").alias("RELienComments"),
            eaiSIG_REPW.businessdate.cast("Date").alias("BusinessDate"),
        )

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_info_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)